In [ ]:
# !pip uninstall transformers -y
!pip install transformers

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel
import torch
from PIL import Image
import matplotlib.pyplot as plt
import json

In [ ]:
MODEL_NAME = "mjawadazad2321/donut-base-Medical_Handwritten_Prescriptions_Information_Extraction_updated"

processor = DonutProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Device:", device)

In [ ]:
image_path = "image.png"  # change this

image = Image.open(image_path).convert("RGB")

plt.imshow(image)
plt.axis("off")

In [ ]:
# resize for better performance
image = image.resize((1280, 1280))

In [ ]:
task_prompt = "<s>"  # try alternatives later if needed

inputs = processor(
    image,
    task_prompt,
    return_tensors="pt"
).to(device)

outputs = model.generate(
    **inputs,
    max_length=512
)

result = processor.batch_decode(outputs, skip_special_tokens=True)[0]

print("RAW OUTPUT:\n")
print(result)

In [ ]:
try:
    data = json.loads(result)
    print("PARSED JSON:\n", data)
except:
    print("Not JSON format")

In [ ]:
def extract_drugs(text):
    lines = text.split("\n")
    
    drugs = []
    for line in lines:
        line = line.strip()
        
        if len(line) > 2:
            drugs.append(line)
    
    return drugs

drugs = extract_drugs(result)

print("\nEXTRACTED DRUGS:")
for d in drugs:
    print("-", d)